# 강의 03 · 실습 3 — RAG 검색기 구축 · (2-1) 빈칸 채우기 I

## 1. 문제상황

- 구름월드 놀이공원 고객센터에는 FAQ 항목이 32개 있습니다.
- 손님 질문은 매번 표현이 다릅니다. 「자유이용권 환불이 되나요」「표를 취소하면 돈을 돌려주나요」는 같은 FAQ 항목을 묻는 질문입니다.
- 담당자는 질문을 받을 때마다 FAQ 목록을 훑어 어느 항목이 답인지 찾아 줍니다.
- 언어 모델에 FAQ 없이 그대로 물으면 구름월드의 규정을 모르므로 일반적인 답을 지어냅니다.
- FAQ에 없는 질문(예: 파이썬 정렬)에는 모른다고 답해야 하는데, 지금은 그 경계를 사람이 판단합니다.

## 2. 문제와 목표

- **문제**: 표현이 다른 질문을 FAQ 항목에 대응시키는 일과, FAQ 밖 질문을 가려내는 일을 사람이 합니다.
- **목표**: FAQ를 미리 벡터 저장소에 정리해 두고, 질문이 오면 표현이 달라도 가장 가까운 항목을 거리 점수와 함께 찾아 주며, 점수가 임계값을 넘으면 문서 밖 질문으로 보고 고정 안내 문장으로 보내는 검색기를 만듭니다.
    - 임계값: 1.5. 점수는 거리라서 작을수록 가깝고, 1위 점수가 임계값 이하이면 통과, 넘으면 컷입니다.
    - 고정 안내 문장: 「문서에서 근거를 찾지 못했습니다. 안내 창구로 문의해 주세요.」
    - FAQ 파일: `day05_faq_구름월드.csv`(32행). 저장 디렉터리: `chroma_db`.
- **목표 달성 여부의 판정 기준**: 적재 문서 수가 32이고, 문서 안 질문(「자유이용권 환불이 되나요?」)의 1위 점수가 임계값 1.5 이하로 통과하며, 문서 밖 질문(「파이썬 리스트 정렬은 어떻게 하나요?」)의 1위 점수가 1.5를 넘어 컷되고 고정 안내 문장이 나오는 것을 실행 기록에서 확인합니다.

## 3. 워크플로우 다이어그램

![워크플로우 다이어그램](imgs/lec03_ex03_s1_diagram.svg)

## 4. 단계별 요구사항

1. **문서를 적재합니다.**
    - `day05_faq_구름월드.csv`를 `csv.DictReader`로 읽고(`utf-8-sig`), `Question`이 빈 행은 버립니다.
    - 행마다 `[카테고리] Q: 질문\nA: 답변` 형식의 본문과 `{"row": 행 번호, "category": 카테고리}` 메타데이터를 가진 `Document`를 만들어 리스트 `docs`에 모으고, 적재 문서 수를 출력합니다.
2. **임베딩을 준비합니다.**
    - `OpenAIEmbeddings(model="text-embedding-3-small")`로 임베딩 부품 `emb`를 만들고, 문장 하나를 벡터로 바꿔 벡터의 길이를 출력합니다.
3. **저장소를 구축하고 영속합니다.**
    - `Chroma.from_documents(docs, emb, persist_directory="chroma_db", ids=[...])`로 저장소 `db`를 만듭니다.
    - 문서 id는 `row-<행 번호>`로 주어 셀을 다시 실행해도 항목이 늘지 않게 합니다.
    - 저장된 항목 수를 출력합니다.
4. **점수와 함께 검색합니다.**
    - 「자유이용권 환불이 되나요?」를 `db.similarity_search_with_score(q, k=3)`으로 검색해 상위 청크 3개의 점수와 본문 앞부분을 출력합니다.
    - 점수는 거리이므로 작을수록 가깝습니다.
5. **임계값으로 컷합니다.**
    - `THRESHOLD = 1.5`와 고정 안내 문장 `NO_EVIDENCE = "문서에서 근거를 찾지 못했습니다. 안내 창구로 문의해 주세요."`를 둡니다.
    - 질문을 받아 1위 점수가 임계값 이하이면 그 청크의 본문을, 넘으면 `NO_EVIDENCE`를 돌려주는 함수 `answer_or_cut`을 만들고, 문서 안 질문과 문서 밖 질문(「파이썬 리스트 정렬은 어떻게 하나요?」)을 넣어 점수·판정·결과를 출력합니다.
    - 고정 안내 문장이 임계값 컷의 「다른 경로」입니다.
    - 판정은 「통과」 또는 「컷」으로 표시합니다.

## 5. 코드 골격 — RAG 인덱싱·검색 5단

RAG 검색기를 세우는 순서는 다음 다섯 단계입니다. 아래 「6. 코드 — 스텝바이스텝」의 코드 셀이 이 다섯 단계와 하나씩 대응합니다. ⑤는 검색 결과에 조건을 거는 독립된 단계입니다.

| 단계 | 하는 일 | 사용하는 코드 | 대응하는 요구사항 |
|---|---|---|---|
| ① 문서 적재 | 원본 파일을 읽어 검색 단위 문서로 만듭니다 | `Document(page_content=..., metadata=...)` | 1 |
| ② 임베딩 준비 | 문장을 숫자 벡터로 바꿀 모델을 지정합니다 | `OpenAIEmbeddings(model="text-embedding-3-small")` | 2 |
| ③ 저장소 구축·영속 | 문서와 임베딩을 넣어 저장소를 만들고 디렉터리에 남깁니다 | `Chroma.from_documents(docs, emb, persist_directory=...)` | 3 |
| ④ 점수 동반 검색 | 질문을 넣어 가까운 문서와 그 거리 점수를 함께 받습니다 | `db.similarity_search_with_score(q, k=3)` | 4 |
| ⑤ 임계값 컷 | 점수가 기준을 넘으면 문서 근거를 쓰지 않고 다른 경로로 보냅니다 | `if s <= THRESHOLD` | 5 |

## 6. 코드 — 스텝바이스텝

### 단계 0 — 준비

라이브러리를 불러오고 API 키를 읽습니다. 임베딩 모델도 OpenAI API를 쓰므로 같은 키가 필요합니다.

- API 키는 `.env` 파일에서 읽습니다.
- `.env` 파일은 실습 루트 폴더(`agentic-ai`)에 한 개만 둡니다. `find_dotenv()`가 노트북 위치에서 상위 폴더로 올라가며 찾습니다.
- `.env` 파일에는 다음 한 줄만 넣습니다.

```
OPENAI_API_KEY=발급받은_키
```

In [ ]:
import csv
import os

from dotenv import ____, ____

from langchain_chroma import ____
from langchain_core.documents import ____
from langchain_openai import ____

load_dotenv(find_dotenv(usecwd=True))
if not os.environ.get("OPENAI_API_KEY"):
    raise SystemExit("agentic-ai 폴더의 .env 파일에 OPENAI_API_KEY 한 줄을 넣습니다.")
print("준비를 마쳤습니다.")

### 단계 ① — 문서 적재 (요구사항 1)

- 검색 단위는 `Document`입니다. 본문(`page_content`)이 임베딩되어 검색에 쓰이고, 메타데이터(`metadata`)는 청크와 함께 돌아와 출처 표시나 필터에 쓰입니다.
- FAQ 한 행을 청크 하나로 삼습니다. 질문과 답을 한 본문에 넣어야 질문 표현으로 검색해도 답이 함께 돌아옵니다.

In [ ]:
CSV_PATH = "day05_faq_구름월드.csv"

docs = []
with open(CSV_PATH, encoding="utf-8-sig", newline="") as f:
    for i, row in enumerate(csv.DictReader(f), start=1):
        if not (row.get("Question") or "").strip():
            ____
        text = ____
        docs.append(Document(page_content=____, metadata=____))

print("적재 문서 수:", len(docs))
print("첫 문서:", docs[0].page_content[:60], "| 메타데이터:", docs[0].metadata)

### 단계 ② — 임베딩 준비 (요구사항 2)

- 임베딩은 문장을 숫자 벡터로 바꾸는 모델입니다. 의미가 가까운 문장은 벡터도 가깝습니다.
- 문서를 넣을 때와 질문을 넣을 때 같은 임베딩을 써야 같은 좌표계에서 거리를 잴 수 있습니다.

In [ ]:
emb = OpenAIEmbeddings(model=____)

vec = emb.embed_query("자유이용권 환불이 되나요?")
print("벡터 길이:", len(vec), "| 앞 세 값:", [round(v, 4) for v in vec[:3]])

### 단계 ③ — 저장소 구축·영속 (요구사항 3)

- `Chroma.from_documents`가 문서마다 임베딩을 계산해 저장소에 넣습니다. `persist_directory`를 주면 디렉터리에 남아 프로그램이 끝나도 유지됩니다.
- 문서 id를 행 번호로 주면 같은 셀을 다시 실행해도 같은 id에 덮어써 항목이 늘지 않습니다.

In [ ]:
db = Chroma.from_documents(
    ____, ____,
    persist_directory=____,
    ids=[____ for d in docs],
)

print("저장된 항목 수:", len(db.get()["ids"]))

### 단계 ④ — 점수 동반 검색 (요구사항 4)

- `similarity_search_with_score`는 청크와 거리 점수를 짝으로 돌려줍니다. 점수는 작을수록 가깝습니다.
- 검색은 언제나 k개를 돌려줍니다. 문서에 답이 없어도 가장 가까운 청크가 나오므로, 점수를 봐야 관계있는 청크인지 알 수 있습니다.

In [ ]:
q = "자유이용권 환불이 되나요?"
print(f"[질문] {q}")
for d, s in ____:
    print(f"  score={s:.4f} | {d.page_content[:60].replace(chr(10), ' / ')}")

### 단계 ⑤ — 임계값 컷 (요구사항 5)

- 임계값은 문서 안 질문의 점수 분포와 문서 밖 질문의 점수 분포 사이에 긋는 선입니다. 여기서는 1.5를 씁니다.
- 컷된 질문을 보낼 곳을 정해야 설계가 닫힙니다. 이 실습에서는 고정 안내 문장으로 보냅니다.

In [ ]:
THRESHOLD = 1.5
NO_EVIDENCE = "문서에서 근거를 찾지 못했습니다. 안내 창구로 문의해 주세요."


def answer_or_cut(question: str) -> str:
    """1위 점수가 임계값 이하이면 청크 본문을, 넘으면 고정 안내 문장을 돌려준다.

    입력: question(질문 문자열)
    기능: k=1로 점수 동반 검색을 하고, 점수와 판정(통과/컷)을 출력한다.
    반환: 통과면 1위 청크의 본문, 컷이면 NO_EVIDENCE
    """
    # 여기에 검색과 판정을 작성합니다.
    return ____


# 실행 부분에는 골격이 없습니다. 「4. 단계별 요구사항」의 5번을 보고 처음부터 작성합니다.
# 여기에 문서 안 질문과 문서 밖 질문을 넣어 결과를 출력하는 반복문을 작성합니다.

## 7. 실행 결과 확인

셀을 위에서 아래로 모두 실행한 뒤 다음 세 가지를 확인합니다.

1. 단계 ①의 「적재 문서 수」가 32이고, 단계 ③의 「저장된 항목 수」도 32입니다. 단계 ③을 다시 실행해도 32로 유지됩니다.
2. 단계 ④에서 「자유이용권 환불이 되나요?」의 상위 청크 3개 중 1위가 `[티켓] Q: 자유이용권 환불 규정…` 청크이고 점수가 1.5보다 작습니다.
3. 단계 ⑤에서 문서 안 질문은 「통과」와 청크 본문이, 문서 밖 질문은 「컷」과 고정 안내 문장이 찍힙니다.

세 가지가 모두 확인되면 완성입니다. 하나라도 다르면 `lec03_ex03_s1.ipynb`와 대조해 채운 빈칸을 고칩니다.